# GSPO: STEM Reasoning via Verifiable Rewards

This notebook applies **GSPO** (Group Sequence Policy Optimization) with **Dr. GRPO**
rewards to improve STEM reasoning quality using verifiable reward functions.

**Training pipeline stage:** 3 of 4 (SFT -> SimPO -> **GSPO** -> STaR)

**Key features:**
- `importance_sampling_level="sequence"` — GSPO sequence-level importance ratios (used in Qwen3 training)
- `loss_type="dr_grpo"` — Dr. GRPO variant (removes reward normalization for more stable training)
- `beta=0.0` — GSPO does not use KL regularization
- `epsilon=3e-4`, `epsilon_high=4e-4` — GSPO asymmetric clipping (from paper v2, section 5.1)
- G=8 completions per prompt for group-relative advantage estimation
- Two-phase training:
  - **Phase 3a:** Verifiable rewards only (math correctness, physics units, chemistry balance)
  - **Phase 3b:** Add conceptual rewards with Reasoning-as-Reward (RaR)
- Domain-specific reward functions via `training.scripts.stem_rewards`

**Reference:** [GSPO paper (arXiv 2507.18071)](https://arxiv.org/abs/2507.18071)

**Domains:** Mathematics, Physics, Chemistry, Biology, Computer Science

In [ ]:
# Install dependencies
!pip install -q unsloth trl peft transformers datasets
!pip install -q accelerate bitsandbytes sentencepiece protobuf
!pip install -q sympy chempy  # For verifiable reward functions

In [ ]:
# ============================================================
# Configuration
# ============================================================

# Model
BASE_MODEL = "unsloth/Qwen3-4B"
SIMPO_CHECKPOINT = "/content/drive/MyDrive/MITS/checkpoints/simpo_qwen3_4b/final_adapter"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/gspo_qwen3_4b"

# GSPO hyperparameters (from Qwen3 paper + GSPO paper v2 section 5.1)
IMPORTANCE_SAMPLING_LEVEL = "sequence"  # GSPO: sequence-level ratios
LOSS_TYPE = "dr_grpo"                  # Dr. GRPO: no reward normalization
SCALE_REWARDS = False                  # Dr. GRPO
BETA = 0.0                             # GSPO: no KL regularization
EPSILON = 3e-4                         # GSPO asymmetric clip lower
EPSILON_HIGH = 4e-4                    # GSPO asymmetric clip upper
G = 8                                  # Number of completions per prompt
MAX_COMPLETION = 2048                  # Maximum completion tokens
MAX_PROMPT_LENGTH = 512
LEARNING_RATE = 5e-6

# Training
BATCH_SIZE = 2         # Per-device (actual = BATCH_SIZE * G completions)
GRADIENT_ACCUMULATION_STEPS = 8
STEPS_PER_GENERATION = 4              # GSPO: partition rollout batch into mini-batches
TOTAL_STEPS_PHASE_A = 500             # Verifiable rewards only
TOTAL_STEPS_PHASE_B = 300             # + conceptual rewards
LOGGING_STEPS = 5
SAVE_STEPS = 100

# Paths
GRPO_PROBLEMS_PATH = "training/data/gspo_problems.jsonl"

# Domains
DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]

print(f"Base model: {BASE_MODEL}")
print(f"SimPO checkpoint: {SIMPO_CHECKPOINT}")
print(f"GSPO: importance_sampling={IMPORTANCE_SAMPLING_LEVEL}, loss={LOSS_TYPE}")
print(f"GSPO: beta={BETA}, epsilon={EPSILON}/{EPSILON_HIGH}")
print(f"Dr. GRPO: scale_rewards={SCALE_REWARDS}, G={G}")
print(f"LR: {LEARNING_RATE}")
print(f"Phase 3a: {TOTAL_STEPS_PHASE_A} steps, Phase 3b: {TOTAL_STEPS_PHASE_B} steps")

In [ ]:
# ============================================================
# Mount Google Drive (optional) and load SimPO checkpoint
# ============================================================
import json
import os
import sys
from collections import Counter, defaultdict

# Try mounting Drive; fall back to local
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive mounted successfully")
except Exception as e:
    print(f"Drive mount failed ({e}), using local storage")
    OUTPUT_DIR = "/content/checkpoints/gspo_qwen3_4b"
    SIMPO_CHECKPOINT = "/content/checkpoints/simpo_qwen3_4b/final_adapter"

# Verify SimPO checkpoint exists
assert os.path.exists(SIMPO_CHECKPOINT), f"SimPO checkpoint not found at {SIMPO_CHECKPOINT}"
print(f"SimPO checkpoint verified: {SIMPO_CHECKPOINT}")

# Add project root to path for importing reward functions
if DRIVE_MOUNTED:
    project_root = "/content/drive/MyDrive/MITS"
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# Load GSPO problems from HuggingFace
# ============================================================

from datasets import load_dataset

print("Loading GSPO problems from Siesher/mits-stem-training-data...")
hf_ds = load_dataset("Siesher/mits-stem-training-data", "gspo")

# Combine train+test for GSPO (we use all problems)
problems = [dict(r) for r in hf_ds["train"]] + [dict(r) for r in hf_ds["test"]]
print(f"Loaded {len(problems)} GSPO problems")

# Categorize problems
verifiable_problems = [p for p in problems if p.get("type", "verifiable") == "verifiable"]
conceptual_problems = [p for p in problems if p.get("type") == "conceptual"]

print(f"  Verifiable: {len(verifiable_problems)}")
print(f"  Conceptual: {len(conceptual_problems)}")

domain_counts = Counter(p.get("domain", "unknown") for p in problems)
for domain, count in sorted(domain_counts.items()):
    print(f"  {domain}: {count} problems")

In [ ]:
# ============================================================
# Import reward functions from training.scripts.stem_rewards
# ============================================================

try:
    from training.scripts.stem_rewards import make_reward_fn
    print("Imported make_reward_fn from training.scripts.stem_rewards")
except ImportError:
    print("WARNING: Could not import stem_rewards, defining fallback reward functions")

    import sympy
    import re

    def make_reward_fn(domain, reward_type="verifiable"):
        """Create a domain-specific reward function.

        Returns a function that takes (prompt, completion, answer) and returns a float reward.
        """

        def extract_boxed_answer(text):
            """Extract answer from \\boxed{...} or final numeric answer."""
            boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
            if boxed:
                return boxed[-1].strip()
            # Try extracting last number
            numbers = re.findall(r"[-+]?\d*\.?\d+", text)
            return numbers[-1] if numbers else ""

        def math_reward(prompt, completion, answer):
            """Verify math answer using SymPy equivalence."""
            extracted = extract_boxed_answer(completion)
            if not extracted or not answer:
                return 0.0
            try:
                pred = sympy.sympify(extracted)
                gold = sympy.sympify(answer)
                if sympy.simplify(pred - gold) == 0:
                    return 1.0
                return 0.0
            except (sympy.SympifyError, TypeError, ValueError):
                return 1.0 if extracted.strip() == str(answer).strip() else 0.0

        def physics_reward(prompt, completion, answer):
            """Verify physics answer (numeric + units check)."""
            extracted = extract_boxed_answer(completion)
            if not extracted or not answer:
                return 0.0
            # Check numeric equivalence with tolerance
            try:
                pred_num = float(re.findall(r"[-+]?\d*\.?\d+", extracted)[0])
                gold_num = float(re.findall(r"[-+]?\d*\.?\d+", str(answer))[0])
                if abs(pred_num - gold_num) / max(abs(gold_num), 1e-10) < 0.05:
                    return 1.0
                return 0.0
            except (ValueError, IndexError):
                return 1.0 if extracted.strip() == str(answer).strip() else 0.0

        def chemistry_reward(prompt, completion, answer):
            """Verify chemistry answer (equation balance, formulas)."""
            extracted = extract_boxed_answer(completion)
            if not extracted or not answer:
                return 0.0
            # Simple string match for now; chempy check if available
            if extracted.strip().lower() == str(answer).strip().lower():
                return 1.0
            return 0.0

        def generic_reward(prompt, completion, answer):
            """Generic reward for biology/CS (keyword matching + structure)."""
            extracted = extract_boxed_answer(completion)
            if extracted.strip().lower() == str(answer).strip().lower():
                return 1.0
            # Partial credit for containing key terms
            answer_lower = str(answer).lower()
            completion_lower = completion.lower()
            if answer_lower in completion_lower:
                return 0.5
            return 0.0

        def conceptual_reward(prompt, completion, answer):
            """Reward for conceptual questions (reasoning quality heuristics)."""
            score = 0.0
            # Reward step-by-step reasoning
            if any(marker in completion.lower() for marker in ["step 1", "first,", "let's think", "because"]):
                score += 0.3
            # Reward Socratic style (asking questions)
            if "?" in completion:
                score += 0.2
            # Reward completeness
            if len(completion.split()) > 50:
                score += 0.2
            # Reward if answer keyword is present
            if answer and str(answer).lower() in completion.lower():
                score += 0.3
            return min(score, 1.0)

        if reward_type == "conceptual":
            return conceptual_reward

        reward_map = {
            "math": math_reward,
            "physics": physics_reward,
            "chemistry": chemistry_reward,
            "biology": generic_reward,
            "cs": generic_reward,
        }
        return reward_map.get(domain, generic_reward)

# Build domain reward functions
verifiable_reward_fns = {d: make_reward_fn(d, "verifiable") for d in DOMAINS}
conceptual_reward_fns = {d: make_reward_fn(d, "conceptual") for d in DOMAINS}
print("Reward functions ready for all domains")

In [ ]:
# ============================================================
# GRPOTrainer setup - Phase 3a and Phase 3b
# ============================================================
import torch
from unsloth import FastLanguageModel
from peft import PeftModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import random

# Load model + SimPO adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_COMPLETION + MAX_PROMPT_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = PeftModel.from_pretrained(model, SIMPO_CHECKPOINT)
model = model.merge_and_unload()
print(f"Loaded SimPO checkpoint from {SIMPO_CHECKPOINT}")

# Apply fresh LoRA for GRPO
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")


def create_grpo_reward_fn(reward_fns_map, problems_list):
    """Create a unified reward function for GRPOTrainer.

    The function takes prompts and completions, looks up the corresponding
    problem metadata, and applies the domain-specific reward function.
    """
    # Build prompt -> problem lookup
    prompt_to_problem = {}
    for p in problems_list:
        prompt_to_problem[p["prompt"].strip()] = p

    def reward_fn(prompts, completions, **kwargs):
        rewards = []
        for prompt_text, completion_text in zip(prompts, completions):
            # Look up problem metadata
            problem = prompt_to_problem.get(prompt_text.strip())
            if problem is None:
                rewards.append(0.0)
                continue

            domain = problem.get("domain", "math")
            answer = problem.get("answer", "")
            fn = reward_fns_map.get(domain, reward_fns_map.get("math"))
            reward = fn(prompt_text, completion_text, answer)
            rewards.append(reward)
        return rewards

    return reward_fn


def format_problems_as_dataset(problem_list):
    """Format problems into a Dataset for GRPOTrainer."""
    formatted = []
    for p in problem_list:
        messages = [
            {"role": "system", "content": "You are a STEM tutor. Solve the problem step by step and put your final answer in \\boxed{}."},
            {"role": "user", "content": p["prompt"]},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        formatted.append({"prompt": prompt})
    return Dataset.from_list(formatted)


print("Model and reward functions ready for GRPO training")

In [ ]:
# ============================================================
# Training loop with monitoring
# Phase 3a: verifiable only, Phase 3b: add conceptual with RaR
# ============================================================

training_logs = []


def run_gspo_phase(phase_name, problems_list, reward_fns_map, max_steps, output_subdir):
    """Run a GSPO training phase with monitoring."""
    print(f"\n{'='*60}")
    print(f"GSPO {phase_name}")
    print(f"{'='*60}")
    print(f"  Problems: {len(problems_list)}, Max steps: {max_steps}")

    # Create dataset and reward function
    ds = format_problems_as_dataset(problems_list)
    reward_fn = create_grpo_reward_fn(reward_fns_map, problems_list)

    phase_output = os.path.join(OUTPUT_DIR, output_subdir)
    os.makedirs(phase_output, exist_ok=True)

    gspo_config = GRPOConfig(
        output_dir=phase_output,
        max_steps=max_steps,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        num_generations=G,
        max_completion_length=MAX_COMPLETION,
        max_prompt_length=MAX_PROMPT_LENGTH,
        # ─── GSPO-specific parameters ───
        importance_sampling_level=IMPORTANCE_SAMPLING_LEVEL,  # sequence-level
        loss_type=LOSS_TYPE,       # dr_grpo
        scale_rewards=SCALE_REWARDS,  # False for Dr. GRPO
        beta=BETA,                 # 0.0 for GSPO (no KL)
        epsilon=EPSILON,           # 3e-4 asymmetric clip
        epsilon_high=EPSILON_HIGH, # 4e-4 asymmetric clip
        steps_per_generation=STEPS_PER_GENERATION,  # partition rollout batch
        # ─── Standard parameters ───
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        optim="adamw_8bit",
        max_grad_norm=0.1,
        temperature=0.9,
        seed=42,
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = GRPOTrainer(
        model=model,
        args=gspo_config,
        train_dataset=ds,
        reward_funcs=reward_fn,
        processing_class=tokenizer,
    )

    # Training
    print("Starting GSPO training...")
    result = trainer.train()

    print(f"\n{phase_name} complete!")
    print(f"  Final loss: {result.training_loss:.4f}")

    # Save phase checkpoint
    trainer.save_model(os.path.join(phase_output, "final"))

    phase_log = {
        "phase": phase_name,
        "steps": max_steps,
        "final_loss": result.training_loss,
        "metrics": result.metrics,
    }
    training_logs.append(phase_log)
    return trainer, result


# ---- Phase 3a: Verifiable rewards only ----
trainer_a, result_a = run_gspo_phase(
    phase_name="Phase 3a (Verifiable Only)",
    problems_list=verifiable_problems,
    reward_fns_map=verifiable_reward_fns,
    max_steps=TOTAL_STEPS_PHASE_A,
    output_subdir="phase_3a",
)

# ---- Phase 3b: Add conceptual rewards with RaR ----
combined_reward_fns = {}
for d in DOMAINS:
    v_fn = verifiable_reward_fns[d]
    c_fn = conceptual_reward_fns[d]

    def make_combined(v_fn=v_fn, c_fn=c_fn):
        def combined(prompt, completion, answer):
            v_score = v_fn(prompt, completion, answer)
            c_score = c_fn(prompt, completion, answer)
            return 0.7 * v_score + 0.3 * c_score
        return combined

    combined_reward_fns[d] = make_combined()

all_problems = verifiable_problems + conceptual_problems
random.seed(42)
random.shuffle(all_problems)

trainer_b, result_b = run_gspo_phase(
    phase_name="Phase 3b (Verifiable + Conceptual RaR)",
    problems_list=all_problems,
    reward_fns_map=combined_reward_fns,
    max_steps=TOTAL_STEPS_PHASE_B,
    output_subdir="phase_3b",
)

print("\n" + "="*60)
print("GSPO Training Summary")
print("="*60)
for log in training_logs:
    print(f"  {log['phase']}: loss={log['final_loss']:.4f}")

In [ ]:
# ============================================================
# Per-domain evaluation after training
# ============================================================

print("Running per-domain evaluation...")

FastLanguageModel.for_inference(model)

eval_results = defaultdict(lambda: {"rewards": [], "total": 0, "correct": 0})

# Sample problems for evaluation
eval_sample_size = min(50, len(problems))
eval_problems = random.sample(problems, eval_sample_size)

for problem in eval_problems:
    domain = problem.get("domain", "unknown")
    answer = problem.get("answer", "")

    messages = [
        {"role": "system", "content": "You are a STEM tutor. Solve the problem step by step and put your final answer in \\boxed{}."},
        {"role": "user", "content": problem["prompt"]},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LENGTH).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_COMPLETION,
            temperature=0.7,
            do_sample=True,
            num_return_sequences=1,
        )

    completion = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # Compute reward
    reward_fn = verifiable_reward_fns.get(domain, verifiable_reward_fns.get("math"))
    reward = reward_fn(problem["prompt"], completion, answer)

    eval_results[domain]["rewards"].append(reward)
    eval_results[domain]["total"] += 1
    if reward >= 0.5:
        eval_results[domain]["correct"] += 1

print("\nPer-domain evaluation results:")
overall_rewards = []
for domain in DOMAINS:
    m = eval_results[domain]
    if m["total"] > 0:
        avg_reward = sum(m["rewards"]) / len(m["rewards"])
        accuracy = 100 * m["correct"] / m["total"]
        print(f"  {domain}: avg_reward={avg_reward:.3f}, accuracy={accuracy:.1f}% ({m['correct']}/{m['total']})")
        overall_rewards.extend(m["rewards"])

if overall_rewards:
    print(f"\n  Overall: avg_reward={sum(overall_rewards)/len(overall_rewards):.3f}")

# Save evaluation
eval_path = os.path.join(OUTPUT_DIR, "grpo_eval_metrics.json")
with open(eval_path, "w") as f:
    json.dump({
        "domain_results": {d: {"avg_reward": sum(v["rewards"])/max(len(v["rewards"]),1), "accuracy": v["correct"]/max(v["total"],1), "total": v["total"]} for d, v in eval_results.items()},
        "training_logs": training_logs,
        "config": {"G": G, "KL_coeff": KL_COEFF, "LR": LEARNING_RATE, "scale_rewards": SCALE_REWARDS},
    }, f, indent=2, default=str)
print(f"Evaluation saved to {eval_path}")

In [ ]:
# ============================================================
# Save final adapter
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final GSPO adapter saved to {final_adapter_path}")

# Save training config
config_to_save = {
    "stage": "gspo",
    "variant": "gspo_dr_grpo",
    "base_model": BASE_MODEL,
    "simpo_checkpoint": SIMPO_CHECKPOINT,
    "importance_sampling_level": IMPORTANCE_SAMPLING_LEVEL,
    "loss_type": LOSS_TYPE,
    "scale_rewards": SCALE_REWARDS,
    "beta": BETA,
    "epsilon": EPSILON,
    "epsilon_high": EPSILON_HIGH,
    "G": G,
    "max_completion": MAX_COMPLETION,
    "learning_rate": LEARNING_RATE,
    "steps_per_generation": STEPS_PER_GENERATION,
    "phase_3a_steps": TOTAL_STEPS_PHASE_A,
    "phase_3b_steps": TOTAL_STEPS_PHASE_B,
    "total_problems": len(problems),
    "verifiable_problems": len(verifiable_problems),
    "conceptual_problems": len(conceptual_problems),
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config_to_save, f, indent=2)
print(f"Config saved to {config_path}")

print("\nDone! GSPO adapter is ready for STaR self-improvement (next stage).")